In [73]:
import numpy as np
import os 
os.environ["KERAS_BACKEND"]="tensorflow"
import keras
from keras import layers
import random

In [70]:
import pathlib
import os
os.environ["KERAS_BACKEND"] = "tensorflow"
import keras

zip_path= keras.utils.get_file(
    origin=(
        "http://storage.googleapis.com/download.tensorflow.org/data/spa-eng.zip"
    ),
    fname="spa-eng",
    extract=True
)
text_path= pathlib.Path(zip_path)/"spa-eng"/"spa.txt"

In [71]:
with open(text_path) as f:
    lines= f.read().split("\n")[:-1]
text_pairs=[]
for line in lines:
    english, spanish= line.split("\t")
    spanish="[start] " + spanish + " [end]"
    text_pairs.append((english,spanish))

In [105]:
random.shuffle(text_pairs)
val_samples= int(0.15*len(text_pairs))
train_samples= len(text_pairs)- 2*val_samples
train_pairs= text_pairs[:train_samples]
val_pairs= text_pairs[train_samples: train_samples+ val_samples]
test_pairs= text_pairs[train_samples+ val_samples:]

In [75]:
import string
import re
import tensorflow as tf
from keras import layers
strip_chars= string.punctuation +"¿"
strip_chars= strip_chars.replace("[","")
strip_chars= strip_chars.replace("]","")

def custom_standardization(input_string):
    lowercase= tf.strings.lower(input_string)
    return tf.strings.regex_replace(
        lowercase, f"[{re.escape(strip_chars)}]",""
    )
vocab_size=15000
sequence_length=20

english_tokenizer= layers.TextVectorization(
    max_tokens=vocab_size,
    output_mode="int",
    output_sequence_length=sequence_length,
    standardize=custom_standardization,
)
spanish_tokenizer= layers.TextVectorization(
    max_tokens=vocab_size,
    output_mode="int",
    output_sequence_length=sequence_length+1,
    standardize=custom_standardization,
)
train_english_texts= [pair[0] for pair in train_pairs]
train_spanish_texts= [pair[1] for pair in train_pairs]
english_tokenizer.adapt(train_english_texts)
spanish_tokenizer.adapt(train_spanish_texts)

    

In [76]:
batch_size=64

def format_dataset(eng,spa):
    eng= english_tokenizer(eng)
    spa= spanish_tokenizer(spa)
    features={"english":eng, "spanish":spa[:,:-1]}
    labels= spa[:,1:]
    sample_weoghts= labels != 0
    return features,labels, sample_weoghts

def make_dataset(pairs):
    eng_texts, spa_texts= zip(*pairs)
    eng_texts= list(eng_texts)
    spa_texts=list(spa_texts)
    dataset= tf.data.Dataset.from_tensor_slices((eng_texts,spa_texts))
    dataset= dataset.batch(batch_size)
    dataset= dataset.map(format_dataset, num_parallel_calls=4)
    return dataset.shuffle(2048).cache()

train_ds= make_dataset(train_pairs)
val_ds= make_dataset(val_pairs)

In [40]:
# b --> batch size
# t --> target_length
# s --> source_length
# d --> vector_size

In [66]:
class TransformerEncoder(keras.Layer):
    def __init__(self,hidden_dim, intermediate_dim, num_heads):
        super().__init__()
        key_dim= hidden_dim// num_heads
        self.self_attention= layers.MultiHeadAttention(num_heads,key_dim)
        self.self_attention_layernorm= layers.LayerNormalization()
        self.feed_forward_1= layers.Dense(intermediate_dim,activation="relu")
        self.feed_forward_2= layers.Dense(hidden_dim)
        self.feed_forward_layernorm= layers.LayerNormalization()

    def call(self,source, source_mask):
        residual= x=source
        mask= source_mask[:,None,:]
        x= self.self_attention(query= x, key=x, value=x, attention_mask=mask)
        x= x+residual
        x= self.self_attention_layernorm(x)
        residual=x
        x= self.feed_forward_1(x)
        x= self.feed_forward_2(x)
        x=x+residual
        x=self.feed_forward_layernorm(x)
        return x
        

In [67]:
class TransformerDecoder(keras.Layer):
    def __init__(self,hidden_dim, intermediate_dim, num_heads):
        super().__init__()
        key_dim= hidden_dim// num_heads
        self.self_attention= layers.MultiHeadAttention(num_heads,key_dim)
        self.self_attention_layernorm= layers.LayerNormalization()
        self.cross_attention= layers.MultiHeadAttention(num_heads,key_dim)
        self.cross_attention_layernorm= layers.LayerNormalization()
        self.feed_forward_1= layers.Dense(intermediate_dim,activation="relu")
        self.feed_forward_2= layers.Dense(hidden_dim)
        self.feed_forward_layernorm= layers.LayerNormalization()

    def call(self,target,source, source_mask):
        residual= x=target
        mask= source_mask[:,None,:]
        x= self.self_attention(query= x, key=x, value=x, attention_mask=mask)
        x= x+residual
        x= self.self_attention_layernorm(x)
        residual=x
        mask= source_mask[:,None,:]
        x= self.cross_attention(query=x,key=source, value=source,attention_mask=mask)
        x=x+residual
        x=self.cross_attention_layernorm(x)
        residual=x
        x= self.feed_forward_1(x)
        x= self.feed_forward_2(x)
        x=x+residual
        x=self.feed_forward_layernorm(x)
        return x
       

In [85]:
from keras import ops

class PositionalEmbedding(keras.Layer):
    def __init__(self, sequence_length, input_dim, output_dim):
        super().__init__()
        self.token_embeddings= layers.Embedding(input_dim, output_dim)
        self.position_embeddings= layers.Embedding(sequence_length, output_dim)

    def call(self,inputs):
        positions= ops.cumsum(ops.ones_like(inputs),axis=-1)-1
        embedded_tokens= self.token_embeddings(inputs)
        embedded_positions= self.position_embeddings(positions)
        return embedded_tokens+ embedded_positions

In [86]:
hidden_dim=256
intermediate_dim=2048
num_heads=8
vocab_size=15000

source= keras.Input(shape=(None,), dtype="int32",name="english")
x=PositionalEmbedding(sequence_length,vocab_size, hidden_dim)(source)
encoder_output= TransformerEncoder(hidden_dim, intermediate_dim, num_heads)(
    source=x,
    source_mask= source !=0
)
target= keras.Input(shape=(None,),dtype="int32", name="spanish")
x= PositionalEmbedding(sequence_length,vocab_size, hidden_dim)(target)
x= TransformerDecoder(hidden_dim, intermediate_dim, num_heads)(
    target=x,
    source= encoder_output,
    source_mask= source !=0
)
x= layers.Dropout(0.5)(x)
target_predictions= layers.Dense(vocab_size, activation="softmax")(x)
transformer= keras.Model([source,target], target_predictions)
transformer.summary()

Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ english             │ (None, None)      │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ positional_embeddi… │ (None, None, 256) │  3,845,120 │ english[0][0]     │
│ (PositionalEmbeddi… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ not_equal_2         │ (None, None)      │          0 │ english[0][0]     │
│ (NotEqual)          │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ spanish             │ (None, None)      │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ transformer_encode… │ (None, None, 256) │  1,315,072 │ positional_embed… │
│ (TransformerEncode… │                   │            │ not_equal_2[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ not_equal_3         │ (None, None)      │          0 │ english[0][0]     │
│ (NotEqual)          │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ positional_embeddi… │ (None, None, 256) │  3,845,120 │ spanish[0][0]     │
│ (PositionalEmbeddi… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ transformer_decode… │ (None, None, 256) │  1,578,752 │ transformer_enco… │
│ (TransformerDecode… │                   │            │ not_equal_3[0][0… │
│                     │                   │            │ positional_embed… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_7 (Dropout) │ (None, None, 256) │          0 │ transformer_deco… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_9 (Dense)     │ (None, None,      │  3,855,000 │ dropout_7[0][0]   │
│                     │ 15000)            │            │                   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 14,439,064 (55.08 MB)

 Trainable params: 14,439,064 (55.08 MB)

 Non-trainable params: 0 (0.00 B)

In [87]:
transformer.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    weighted_metrics=["accuracy"],
)
transformer.fit(train_ds,epochs=30, validation_data=val_ds)

Epoch 1/30
1302/1302 ━━━━━━━━━━━━━━━━━━━━ 552s 423ms/step - accuracy: 0.6714 - loss: 0.8756 - val_accuracy: 0.8575 - val_loss: 0.3538
Epoch 2/30
1302/1302 ━━━━━━━━━━━━━━━━━━━━ 601s 461ms/step - accuracy: 0.8629 - loss: 0.3359 - val_accuracy: 0.8951 - val_loss: 0.2424
Epoch 3/30
1302/1302 ━━━━━━━━━━━━━━━━━━━━ 648s 498ms/step - accuracy: 0.8961 - loss: 0.2278 - val_accuracy: 0.9098 - val_loss: 0.2035
Epoch 4/30
1302/1302 ━━━━━━━━━━━━━━━━━━━━ 655s 503ms/step - accuracy: 0.9152 - loss: 0.1706 - val_accuracy: 0.9162 - val_loss: 0.1909
Epoch 5/30
1302/1302 ━━━━━━━━━━━━━━━━━━━━ 615s 472ms/step - accuracy: 0.9265 - loss: 0.1371 - val_accuracy: 0.9133 - val_loss: 0.1935
Epoch 6/30
1302/1302 ━━━━━━━━━━━━━━━━━━━━ 593s 456ms/step - accuracy: 0.9341 - loss: 0.1157 - val_accuracy: 0.9120 - val_loss: 0.1942
Epoch 7/30
1302/1302 ━━━━━━━━━━━━━━━━━━━━ 666s 512ms/step - accuracy: 0.9406 - loss: 0.1001 - val_accuracy: 0.9177 - val_loss: 0.1962
Epoch 8/30
1302/1302 ━━━━━━━━━━━━━━━━━━━━ 654s 502ms/step - ac

In [108]:
import numpy as np

spa_vocab= spanish_tokenizer.get_vocabulary()
spa_index_lookup= dict(zip(range(len(spa_vocab)), spa_vocab))

def generate_translation(input_sentence):
    tokenized_input_sentence= english_tokenizer([input_sentence])
    decoded_sentence="[start]"
    for i in range (sequence_length):
        tokenized_target_sentence= spanish_tokenizer([decoded_sentence])
        tokenized_target_sentence= tokenized_target_sentence[:,:-1]
        inputs= [tokenized_input_sentence, tokenized_target_sentence]
        next_token_predictions=transformer.predict(inputs,verbose=0)
        sampled_token_index= np.argmax(next_token_predictions[0,i,:])
        sampled_token= spa_index_lookup[sampled_token_index]
        decoded_sentence += " " + sampled_token
        if sampled_token == "[end]":
            break
    return decoded_sentence

test_eng_texts=[pair[0] for pair in test_pairs]
for _ in range(5):
    input_sentence= random.choice(test_eng_texts)
    print("-")
    print(input_sentence)
    print(generate_translation(input_sentence))

-
I studied it thoroughly.
[start] arriesgues simpático [end]
-
I know each one of you.
[start] arriesgues rato caer mucho rato [end]
-
I noticed she was wearing a new hat.
[start] arriesgues rato cantante planea favorito [end]
-
Brace yourself.
[start] atmósfera bien [end]
-
A dog bit her leg.
[start] arriesgues baile [end]
